## Cosmic Web - Intro

### Table of Contents

1. [Motivation](#1-motivation)
2. [Density field](#2-density-field)
3. [Why this problem](#3-why-this-problem)
4. [Algorithm](#4-algorithm)
5. [Reference](#5-reference)
6. [Validation](#6-validation)
7. [Sizing](#7-sizing)
8. [Baseline](#8-baseline)
9. [Ladder](#9-ladder)

---

### 1. Motivation

We have looked at array-oriented computing with NumPy and CuPy. Other libraries provide different programming models, and some of them better suit a given scientific problem. In this block we take a computation from [CuPy](https://docs.cupy.dev/en/stable/)'s array model through four more programming models: [JAX](https://docs.jax.dev/en/latest/)'s traced and fused graph, [cuda.compute](https://nvidia.github.io/cccl/python/)'s algorithms over iterators, [numba-cuda-mlir](https://github.com/NVIDIA/numba-cuda-mlir)'s CUDA kernels written in Python, and bindings to existing C++ kernels/libraries with [CppJIT](https://github.com/compiler-research/cppjit).

The computation comes from cosmology and is small enough to understand completely in this notebook.

### 2. Density field

Let's look at a box of points representing matter in a grid, which is given a faint ripple. Gravity pulls every point towards the nearby excess. The Zel'dovich approximation ([Zel'dovich 1970](https://ui.adsabs.harvard.edu/abs/1970A%26A.....5...84Z)) is the simplest model of what follows:

- Each point moves in a straight line in the direction of its first pull.
- The distance grows with time: $D$ times the first displacement, where the growth factor $D$ stands in for time.
- The model holds until the first points cross each other.

The animation follows the particles that started on a plane of the grid, seen from above, as $D$ runs from 0 to 1.


In [ ]:
import numpy as np
import cupy as cp

import cosmic_core as cw

cw.machine_report()
frames = cw.zeldovich_particles(n=128, start=0.0, end=1.0, interval=0.1)
cw.animate_particles(frames, n=128)

- At $D = 0$ the points sit uniformly on the grid.
- As $D$ grows they drift towards the regions that started out denser and pile up along lines.
- In 3D these lines are **sheets**, seen edge-on.
- Sheets meet along **filaments**, and filaments meet at **knots**, seen here as dense spots.
- The empty regions between them are **voids**.

This network of sheets, filaments and knots around voids is a representation of the **cosmic web** ([Bond et al. 1996](https://doi.org/10.1038/380603a0)).

We can now convert the point cloud into a *density field* in four steps, shown on the above frame:

1. **Deposit.** Each grid cell stores the total mass of particles that landed in it.
2. **Contrast.** Divide by the mean and subtract one. The result $\delta$: positive where a cell holds more than the average and negative where it holds less. An empty cell has $\delta = -1$.
3. **Blur.** Smooth $\delta$ over a few cells. This removes the graininess of individual points and retains the structure.
4. **Standardise.** Divide the blurred $\delta$ by its scatter $\sigma$. The value $\nu = \delta/\sigma$ is how many typical fluctuations above or below average the cell sits.


In [ ]:
cw.plot_points_to_field(n=256, growth=1.0)

The resulting field $\nu$ is the kind of data we analyze in this tutorial.

Let's take a full 3D field and apply a threshold $\nu_0$ keeping cells with $\nu \ge \nu_0$. This mask allows us to understand the structures that form the web.

---

**Quick Docs**

- `cw.zeldovich_field(n)`: `(nu, lam, label)`, the field of §2 on an $n^3$ grid; `lam` is the parameter $\lambda$ of the Gaussian formula in §6


In [ ]:
nu, lam, label = cw.zeldovich_field(n=256)
print(f'nu: shape {nu.shape}, dtype {nu.dtype}, mean {nu.mean():.2f}, scatter {nu.std():.2f}')
cw.plot_slices(nu, thresholds=[2, 0, -1])

What do we observe?

- $\nu \ge 2$: only the peaks survive (knots).
- $\nu \ge 0$: roughly half the cells survive as a connected network of sheets and filaments.
- $\nu \ge -1$: nearly everything survives, leaving isolated holes (voids).

How the set is connected changes with the value. Let's try to measure the connectivity as a function of this threshold level.


### 3. Why this problem

Counting how a set of 3D points is connected seems like a global problem: identify every piece, tunnel and cavity.

However, the *Euler characteristic* decomposes it into a local one: sum over grid cells that each require just the immediate neighbouring cell values.

- Apply a 3D stencil, followed by a histogram (both common patterns in HPC):
    - The stencil reads each cell's neighbours.
    - The histogram adds one count per cell to a table.
- The result is an integer, so implementations can be compared exactly, without tolerances.
- A pass over the whole field returns a set of integers that can be plotted as a curve.
- For a Gaussian random field input, the curve is a known solution in closed form (§6): an independent check that our code is correct before we benchmark it.


### 4. Algorithm

Euler's formula on a pixel grid - on the thresholded field:

- Every kept pixel enters our set as a *vertex*.
- Every adjacent vertex is joined as an *edge*.
- Where four vertices form a 2×2 block, join the square as a *face*.

$$\chi = V - E + F$$

Vertices, edges and faces are the *cells* of the mask. The resulting $\chi$ is the Euler characteristic of the mask: the number of connected components minus the number of holes in them.

Let's implement the count for three small masks:


In [ ]:
def euler_2d(mask):
    """(V, E, F, V - E + F) of a pixel mask: pixels, side-by-side pairs, 2x2 blocks."""
    v = mask.sum()
    e = (mask[1:, :] & mask[:-1, :]).sum() + (mask[:, 1:] & mask[:, :-1]).sum()
    f = (mask[1:, 1:] & mask[:-1, 1:] & mask[1:, :-1] & mask[:-1, :-1]).sum()
    return v, e, f, v - e + f

masks = {
    'one pixel': np.array([[0, 0, 0], [0, 1, 0], [0, 0, 0]], dtype=bool),
    '2x2 block': np.array([[0, 0, 0, 0], [0, 1, 1, 0], [0, 1, 1, 0], [0, 0, 0, 0]], dtype=bool),
    'ring':      np.array([[1, 1, 1], [1, 0, 1], [1, 1, 1]], dtype=bool),
}
print(f"{'mask':<10} {'V':>3} {'E':>3} {'F':>3} {'V-E+F':>6}")
for name, m in masks.items():
    v, e, f, chi = euler_2d(m)
    print(f"{name:<10} {v:>3} {e:>3} {f:>3} {chi:>6}")

Now a larger mask with an animation that draws the cells as it counts them:

In [ ]:
mask = np.array([
    [0, 1, 1, 1, 0, 0],
    [0, 1, 0, 1, 0, 1],
    [0, 1, 1, 1, 0, 1],
    [0, 0, 0, 0, 0, 1],
    [1, 1, 0, 0, 0, 0],
    [1, 1, 0, 0, 0, 0],
], dtype=bool)
print('6x6 mask: V-E+F =', euler_2d(mask)[3])
cw.animate_euler_buildup(mask)

#### The curve on one plane

Now let's lower the level across a 64×64 slice and compute $\chi$ at every step.


In [ ]:
slice64 = nu[:64, :64, nu.shape[2] // 2]
cw.animate_flood(slice64, start=2.5, end=-2.5, interval=0.125)

- At the initial levels, $\chi$ counts the blobs seen in the mask.
- As the level falls the blobs merge and holes open inside them: $\chi$ rises, then falls back.

This value summarizes a shape's connectivity and holes, helping computational scientists track structures in simulations and volumetric data.

![ec-illustration](./assets/euler-characteristic-dark.gif)

#### One more dimension

- For ordinary 3D solids, $\chi = \text{connected pieces} - \text{tunnels} + \text{enclosed cavities}$: a solid ball gives $1$, a solid ring $0$, and a hollow shell $2$.

The meshes above are hollow surfaces built from vertices, edges and faces. Our mask is solid, which adds a term. In 3D a grid cell is a *voxel*: eight vertices join as a 2×2×2 *cube* that enters with a negative sign: $\chi = V - E + F - C.$

- Our box is **periodic**: opposite sides connect. Once every voxel is included, $\chi = 0$.
- Each voxel owns **eight** cells extending in the positive directions: one vertex, three edges ($+x,+y,+z$), three faces, and one cube. This counts each cell exactly once.

Let's look at the masks on the data seen in §2 as 3D surfaces:

In [ ]:
cw.isosurfaces(nu, thresholds=[2, 0, -1], sub=48)

#### Births

As the level falls, the threshold at which a cell enters the set and remains is called its "birth". Hence a cell counts at every level at or below its birth:

- Every cell casts one *token* at its birth: $+1$ for a vertex or a face, $-1$ for an edge or a cube.
- Sort the tokens into $K$ bins by birth level.
- $\chi$ at a level is the cumulative sum of the tokens in that bin and every bin above it: $\chi(\nu) = \sum_{\text{cells}} (-1)^{\dim}\,[\,\text{birth} \ge \nu\,]$

where:
- $\dim$ is the cell's dimension (vertex 0, edge 1, face 2, cube 3)
- The bracket is 1 when its condition holds and 0 otherwise.

For the full box $\chi = 0$: the tokens should sum to zero as a consistency check.

Let's try this on a 3×3 patch with one bin per value, and check each level against $V - E + F$.

---

**Quick Docs**

- `cw.plot_birth_example(patch, k)`: illustrates cells that enter at each level, the tokens per bin and $\chi$ at every level; returns `(tokens, chi)`.


In [ ]:
patch = np.array([[3, 1, 3],
                  [1, 0, 1],
                  [3, 1, 3]])
tokens, chi_patch = cw.plot_birth_example(patch, k=4)
for level in range(3, -1, -1):
    v, e, f, chi_direct = euler_2d(patch >= level)
    print(f'level {level}: Tokens summed {chi_patch[level]:>2}  Formula V - E + F = {v} - {e} + {f} = {chi_direct}')

The same on an 8×8×8 region of the field:

In [ ]:
cw.animate_birthdays(nu[:8, :8, :8], k=16)

### 5. Reference

From here on we use a real simulation. The [Quijote simulations](https://quijote-simulations.readthedocs.io/) ([Villaescusa-Navarro et al. 2020](https://doi.org/10.3847/1538-4365/ab9d82)) follow $512^3$ dark-matter particles in a 1000 Mpc/h box under gravity to the present day. Let's deposit the particle positions on a grid with *cloud-in-cell* (CIC) assignment, where each particle shares its mass among the eight nearest cells, then apply the four steps of §2 on the GPU. 

--- 

The data is released under the [MIT License](https://quijote-simulations.readthedocs.io/en/latest/license.html), copyright 2019 Francisco Villaescusa-Navarro.


**Quick Docs**

- `cw.quijote_field()`: `(nu, lam, label)` on a $512^3$ grid, generated with CuPy from the particle positions. The positions are read from `$COSMIC_WEB_DATA` or `data/`.


In [ ]:
nu, lam, label = cw.quijote_field()
print(f'nu: shape {nu.shape}, dtype {nu.dtype}, mean {nu.mean():.2f}, scatter {nu.std():.2f}')
cw.plot_slices(nu, thresholds=[2, 0, -1])

Let's look at a solution implemented with CuPy: `cw.ecc_hist_cupy(q, k)` computes the histogram on the GPU.

- Input: an array `q` of bin indices from $0$ to $K-1$ (read by later notebooks).
- Output: the token histogram, one integer per bin.

```py
def ecc_cells_cupy(q):
    """Births of the eight cells each voxel owns, on the device: (vertex, (ex, ey, ez), (fxy, fxz, fyz), cube)."""
    ex = cp.minimum(q, cp.roll(q, -1, 0))
    ey = cp.minimum(q, cp.roll(q, -1, 1))
    ez = cp.minimum(q, cp.roll(q, -1, 2))
    fxy = cp.minimum(ex, cp.roll(ex, -1, 1))
    fxz = cp.minimum(ex, cp.roll(ex, -1, 2))
    fyz = cp.minimum(ey, cp.roll(ey, -1, 2))
    c = cp.minimum(fxy, cp.roll(fxy, -1, 2))
    return q, (ex, ey, ez), (fxy, fxz, fyz), c

def ecc_hist_cupy(q, k=K_BINS):
    """Signed histogram of cell births on the device, int64 of length k: +1 per vertex and face, -1 per edge or cube."""
    v, (ex, ey, ez), (fxy, fxz, fyz), c = ecc_cells_cupy(q)
    bc = lambda a: cp.bincount(a.ravel(), minlength=k)
    return bc(v) - bc(ex) - bc(ey) - bc(ez) + bc(fxy) + bc(fxz) + bc(fyz) - bc(c)
```

Solution breakdown:

- **Find neighbours.** `cp.roll(q, -1, axis)` aligns each voxel with its next neighbour along that axis. At the boundary, it wraps around to the opposite side of the box.
- **Compute birth bins.** `ecc_cells_cupy` finds the lowest corner bin for each cell: two corners for an edge, four for a face, and eight for a cube. The repeated `cp.minimum` calls build these results from pairs of neighbours.
- **Histogram.** `ecc_hist_cupy` uses `cp.bincount` to count how many cells are born in each bin. It adds the vertex and face counts and subtracts the edge and cube counts.

Binning preserves order: taking the minimum before or after binning gives the same birth bin. This ensures the histogram is exact for the binned field.

Let's run the reference on the Quijote field `nu`.

---

**Quick Docs**

- `cw.quantise(nu)`: the bin index of every voxel in `nu`.
- `cw.bin_edges()`: the lower edge of every bin.
- `cw.ecc_from_hist(hist)`: $\chi$ at every bin edge.


In [ ]:
K = cw.K_BINS
q = cw.quantise(cp.asarray(nu))
edges = cw.bin_edges()
hist = cp.asnumpy(cw.ecc_hist_cupy(q))
chi = cw.ecc_from_hist(hist)
assert hist.sum() == 0, hist.sum()
print(f'N = {nu.shape[0]}^3, K = {K}: sum(tokens) = {hist.sum()}, max |chi| = {np.abs(chi).max()}')
cw.plot_hist_curve(edges, hist, chi)

The histogram is positive at high levels, where mostly vertices are born, and negative in the middle, where edges dominate.

### 6. Validation

The other sweep sizes in every notebook use a Gaussian random test field. The solution to its expected curve $\chi(\nu)$ is known in closed form ([Hamilton et al. 1986](https://doi.org/10.1086/164571)):

$$\langle\chi(\nu)\rangle = V\,\frac{\lambda^{3}}{(2\pi)^{2}}\,(\nu^{2}-1)\,e^{-\nu^{2}/2}, \qquad \lambda^{2} = \tfrac{1}{3}\langle|\nabla\nu|^{2}\rangle,$$

where $V$ is the number of voxels and the gradient $\nabla\nu$ is taken in voxel units. The factor $(\nu^{2}-1)$ causes the curve to cross zero at $\nu = \pm 1$, independent of $\lambda$ or $V$.

Let's generate the test field at the same size and overlay the formula.

---

**Quick Docs**

- `cw.make_binned_field(n)`: `(q, lam)`, the binned test field (int32, on device) and $\lambda$, computed from the field.
- `cw.chi_gaussian(edges, lam, volume)`: the expected $\chi$ at every bin edge.
- `cw.plot_overlay(edges, chi, pred)`: plots the measured $\chi$ over the formula, with their difference. Returns `(rel, zeros)`: the largest difference relative to the solution, and the levels where $\chi$ crosses zero.


In [ ]:
N = nu.shape[0]
q, lam = cw.make_binned_field(N)
chi = cw.ecc_from_hist(cp.asnumpy(cw.ecc_hist_cupy(q)))
pred = cw.chi_gaussian(edges, lam, q.size)
rel, zeros = cw.plot_overlay(edges, chi, pred, title=f'the test field at {N}^3 against the Gaussian expectation')
print(f'relative difference {rel:.3f} (tolerance {cw.ANALYTIC_TOL}), zero crossings at nu = {np.round(zeros, 2).tolist()}')
assert rel < cw.ANALYTIC_TOL, rel

- Both curves cross zero near $\nu = \pm 1$, as the formula predicts.


### 7. Sizing

We select the sweep sizes from the GPU's memory, then measure device copy bandwidth, used as a reference for later benchmarks. Each tool skips sizes where its temporary arrays would exceed device memory.


In [ ]:
cw.save_sizing()

### 8. Baseline

Time the reference histogram at the shared benchmark size and save it to `timings/cupy.json`. Later notebooks save their results alongside it. The field is binned on the GPU, while the cumulative sum runs on the CPU, outside the timing. Would moving it to the GPU help?

- Every tool returns its $K$-value histogram to the host, where a cumulative sum costs less than the kernel launches it would need on the GPU.
- The same step is shared by all tools, so the timed region is where our implementation differs.

---

**Quick Docs**

- `cw.field(n)`: returns the binned $n^3$ field on the GPU, built once per size before timing.
- `cw.measure(stage, run, n, ...)`: times the first call and the median of repeated calls. Ensures an exact match to the reference curve. Saves to `timings/<stage>.json` if `record=True`.
- `cw.save_sweep(stage, run)`: benchmarks sizes that fit in memory, checks results against the reference, and appends them to the stage file. Reports skipped sizes.


In [ ]:
N = cw.benchmark_n()


def run_cupy(n):
    hist = cw.ecc_hist_cupy(cw.field(n), K)
    cp.cuda.get_current_stream().synchronize()
    return cp.asnumpy(hist)


cw.measure('cupy', run_cupy, N)
cw.save_sweep('cupy', run_cupy)

### 9. Ladder

Each notebook implements `ecc_hist_<tool>` using the field already on the GPU. The resulting histogram must match the CuPy reference exactly and sum to zero:

$$\max_b |\chi_{\text{tool}}[b] - \chi_{\text{cupy}}[b]| = 0 \quad\text{and}\quad \sum_b \text{hist}[b] = 0.$$

| NB | Tool | Approach |
|---|---|---|
| 91 | **CuPy** | Array operations that translate to kernels |
| 92 | **JAX** | XLA: JIT compilation and kernel fusion |
| 93 | **cuda.compute** | Iterators compute the stencil inside a library kernel |
| 93 | **numba-cuda-mlir** | A Python CUDA kernel with a shared-memory histogram |
| 94 | **CppJIT** | Interop with CUDA C++ kernels |

`95__cosmic_web__synthesis.ipynb` analyzes the results from `timings/` and compares the resulting curves, timelines, and memory traffic.

Shut down each notebook's Jupyter kernel before starting the next to release its GPU memory. If memory is insufficient, the next notebook stops and reports at its first cell.

Next: `91__cosmic_web__cupy__array_model.ipynb` traces the reference implementation and measures each operation's cost.